In [ ]:
import os
import json
from tqdm import tqdm
from llama_cpp import Llama

# Configuration
MODEL_PATH = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/Phi-4_q8.gguf"
INPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/TiJian94.json"
OUTPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/results_with_phi4.json"

GPU_LAYERS = -1
N_CTX = 32768               
N_BATCH = 1024
N_UBATCH = 512
N_THREADS = 4
MAIN_GPU = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.1
TOP_P = 0.8
SAVE_EVERY = 10

# Load model
print("Loading GGUF Model...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=GPU_LAYERS, n_ctx=N_CTX, n_batch=N_BATCH, n_ubatch=N_UBATCH,
    n_threads=N_THREADS, verbose=False,
)
print("Model Loaded!")

# Read input data
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(f"Total records: {len(dataset)}")

# Resume from checkpoint
results = []
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        results = json.load(f)
    start_idx = len(results)
    print(f"Found existing output file, resuming from record {start_idx}.")
else:
    start_idx = 0

# Inference
for idx in tqdm(range(start_idx, len(dataset)), desc="Inference Progress"):
    item = dataset[idx]
    instruction = item.get("instruction", "")
    input_text = item.get("input", "")
    gt_output = item.get("output", "")

    try:
        # Construct chat messages
        messages = [{"role": "user", "content": instruction + "\n" + input_text}]
        
        # Inference
        response = llm.create_chat_completion(
            messages=messages, temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS
        )
        phi4_output = response["choices"][0]["message"]["content"].strip()
        
        results.append({
            "instruction": instruction, "input": input_text, 
            "output": gt_output, "output_phi4": phi4_output
        })
    except Exception as e:
        print(f"\nInference failed at record {idx}: {e}")
        results.append({
            "instruction": instruction, "input": input_text, 
            "output": gt_output, "output_phi4": "", "error": str(e)
        })

    # Save checkpoint every SAVE_EVERY records
    if len(results) % SAVE_EVERY == 0:
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
        print(f"\nSaved {len(results)} records")

# Final save
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("All inference completed!")
print(f"Results saved to: {OUTPUT_JSON}")

Loading GGUF Model...


llama_context: n_ctx_seq (32768) > n_ctx_train (16384) -- possible training context overflow


Model Loaded!
Total records: 94


推理进度:  11%|█         | 10/94 [10:16<1:25:43, 61.24s/it]


已保存 10 条结果


推理进度:  21%|██▏       | 20/94 [16:30<56:36, 45.90s/it]  

... (13 outputs left out)

推理进度:  96%|█████████▌| 90/94 [1:17:48<03:01, 45.38s/it]


已保存 90 条结果


推理进度: 100%|██████████| 94/94 [1:21:00<00:00, 51.71s/it]

全部推理完成！
结果已保存到：/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/results_with_phi4.json
